In [4]:
import torch
from diffusers import AutoPipelineForText2Image
from PIL import Image
import os

# ==== CONFIG ====
models = {
    "SDXL_turbo": "stabilityai/sdxl-turbo",  # or replace with another IP-compatible model
    "SG161222" : "SG161222/Realistic_Vision_V5.1_noVAE"
}

ip_adapter_path = "h94/IP-Adapter"  # Pretrained IP-Adapter
ip_weights = "ip-adapter_sdxl.bin"  # Inside 'sdxl_models/' subfolder

logo_path = "nike.png"
prompt = "A summer forest banner, bright lighting, detailed landscape"
negative_prompt = "ugly, blurry, deformed, low quality, text, watermark"
guidance_scale = 7.5
steps = 5
adapter_scale = 0.7
output_dir = "results/ip_adapter_comparison"
os.makedirs(output_dir, exist_ok=True)


In [5]:

device = "mps" if torch.backends.mps.is_available() else "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.float16 # torch.float32 if device == "mps" else

print(f"🖥️ Using device: {device} | dtype: {dtype}")

# ==== LOAD LOGO ====
logo = Image.open(logo_path).convert("RGB").resize((224, 224))  # Resize to match IP-Adapter expected input

# ==== LOOP OVER MODELS ====
for name, model_id in models.items():
    print(f"\n🚀 Generating with: {name}")
    
    pipe = AutoPipelineForText2Image.from_pretrained(
        model_id,
        torch_dtype=dtype,
        variant="fp16" if dtype == torch.float16 else None
    ).to(device)

    pipe.load_ip_adapter(ip_adapter_path, subfolder="sdxl_models", weight_name=ip_weights)
    pipe.set_ip_adapter_scale(adapter_scale)
    #pipe.enable_attention_slicing()  # Optional memory improvement

    generator = torch.Generator(device=device).manual_seed(42)

    result = pipe(
        prompt=prompt,
        ip_adapter_image=logo,
        negative_prompt=negative_prompt,
        num_inference_steps=steps,
        generator=generator,
    ).images[0]

    out_path = os.path.join(output_dir, f"{name}_with_logo.png")
    result.save(out_path)
    print(f"✅ Saved: {out_path}")

🖥️ Using device: mps | dtype: torch.float16

🚀 Generating with: SDXL_turbo


Fetching 18 files:  83%|████████▎ | 15/18 [07:55<01:35, 31.70s/it]


KeyboardInterrupt: 